In [4]:
import pandas as pd
import altair as alt
import dataframe_image as dfi

In [5]:
class MWI:
    
    all_rounds = [
        'Round of 64', 
        'Round of 32', 
        'Sweet 16', 
        'Elite 8', 
        'Final 4', 
        'Championship', 
        'Winner'
    ]
    
    def __init__(self, file_path):
        
        self.brackets = pd.read_excel(file_path, sheet_name=None)
        self.actual_df = self.brackets['Actual Results']
        self.current_round = self.get_current_round()
        self.eliminated_teams = self.get_eliminated_teams()        
        self.leaderboard = self.get_leaderboard()
        
        print(f'Current Round: \t\t{self.current_round}')
        print(f'Num. Eliminated Team: \t{len(self.eliminated_teams)}')
        
    def get_eliminated_teams(self):
        
        chunks_by_round_dict = {
            'Round of 32': 2,
            'Sweet 16': 4,
            'Elite 8': 8,
            'Final 4': 16,
            'Championship': 32,
            'Winner': 64,
        }
        
        match_ups = []
        match_up_results = []
        eliminated_teams = set()
        
        
        for r, chunk_size in chunks_by_round_dict.items():
            match_up_results = []
            match_ups = []
            for i in range(0, self.actual_df.shape[0], chunk_size):
                match_up_results.append(self.actual_df[i:i + chunk_size][r].dropna().to_list())
                match_ups.append(self.actual_df[i:i + chunk_size]['Round of 64'].to_list())
                        
            for index, result in enumerate(match_up_results):
                if result:
                    eliminated_teams.update(set(match_ups[index]).difference(set(result)))
                
        return eliminated_teams
        
    def validate_df(self, df):
        
        col_dict = {
            'Round of 32': 32, 
            'Sweet 16': 16, 
            'Elite 8': 8,
            'Final 4': 4,
            'Championship': 2,
            'Winner': 1,
        }
        
        for r, num_teams in col_dict.items():
            assert len(df[r].dropna()) == num_teams
        
    def get_current_round(self):
        
        for r in reversed(self.all_rounds):
            if len(self.actual_df[r].dropna()) > 0:
                return r
            
    def get_leaderboard(self, potential_points=True):
        
        col_map = {
            'Round of 32': 'Round of 64',
            'Sweet 16': 'Round of 32',
            'Elite 8': 'Sweet 16',
            'Final 4': 'Elite 8',
            'Championship': 'Final 4',
            'Winner': 'Championship',
            'Champion': 'Winner',
        }
        
        rounds = [
            ('Round of 32', 10), 
            ('Sweet 16', 20), 
            ('Elite 8', 40), 
            ('Final 4', 80), 
            ('Championship', 160),
            ('Winner', 320)
        ]
        
        results_dict = {
            'MWI Contestant': [],
            'Champion': [],
            'Runner-Up': [],
            'Round of 32': [],
            'Sweet 16': [],
            'Elite 8': [],
            'Final 4': [],
            'Championship': [],
            'Winner': [],
            'Total Points': [],
            'Total Potential Points': []
        }
        
        final_col_order = [
            'Place',
            'MWI Contestant',
            'Winner',
            'Runner-Up',
            'Round of 64', 
            'Round of 32', 
            'Sweet 16', 
            'Elite 8', 
            'Final 4', 
            'Championship', 
            'Total Points',
            'Total Potential Points'
        ]
        
        for sheet_name, contestant_df in self.brackets.items():
            if sheet_name != 'Actual Results' and 'Contestant' not in sheet_name:
            #if sheet_name != 'Actual Results' and sheet_name in ['Shawn', 'Maren', 'Elaine', 'Colombina', 'Roman', 'Eli', 'Pete', 'Letty', 'Obama', 'Sarah', 'Lilli', 'Cynthia', 'Tim', 'Emma', 'Mary', 'Chad', 'Truett', 'Sabine', 'Austin', 'Pia', 'August', 'Don', 'Penny']:
                self.validate_df(contestant_df)
                results_dict['MWI Contestant'].append(sheet_name)
                champ = contestant_df['Winner'].dropna().to_list()
                runner_up = list(set(contestant_df['Championship'].dropna().to_list()) - set(champ))
                results_dict['Champion'].append(champ[0])
                results_dict['Runner-Up'].append(runner_up[0])
                
                total_points = 0
                total_potential_points = 0
                
                for r, points in rounds:
                    y_true = set(self.actual_df[r].dropna().to_list())
                    y_pred = set(contestant_df[r].dropna().to_list())
                    matches = len(y_true.intersection(y_pred))
                    potential_matches = len(y_pred.difference(y_true).difference(self.eliminated_teams))
                    if potential_matches:
                        potential_matches_text = f'({potential_matches * points})'
                    else:
                        potential_matches_text = ''
                    if matches:
                        matches_text = f'{matches * points}'
                    else:
                        matches_text = ''
                    if matches_text + potential_matches_text == '':
                        text = '-'
                    else:
                        text = f'{matches_text} {potential_matches_text}'
                    
                    results_dict[r].append(text)
                    total_points += (matches * points)
                    total_potential_points += ((potential_matches * points) + (matches * points))
                results_dict['Total Points'].append(total_points)
                results_dict['Total Potential Points'].append(total_potential_points)

        df = pd.DataFrame(results_dict).rename(columns=col_map)
        df['Place'] = df['Total Points'].rank(ascending=False, method='min').astype(int)
        
        return (df[final_col_order]
                    .sort_values(by=['Place', 'Winner', 'Runner-Up', 'MWI Contestant'])
                    .reset_index(drop=True))        

In [6]:
970-810

160

In [15]:
mwi = MWI('MWI 2026 Auto v2.xlsx')
mwi.leaderboard

Current Round: 		Sweet 16
Num. Eliminated Team: 	48


,Place,MWI Contestant,Winner,Runner-Up,Round of 64,Round of 32,Sweet 16,Elite 8,Final 4,Championship,Total Points,Total Potential Points
0,1,Truett,(1) Michigan,(1) Duke,280,240,(280),(240),(320),(320),520,1680
1,2,Eli,(1) Duke,(1) Michigan,270,220,(240),(320),(320),(320),490,1690
2,2,Don,(1) Michigan,(1) Duke,250,240,(200),(320),(320),(320),490,1650
3,2,Tim,(2) Iowa St,(1) Duke,270,220,(240),(320),(320),(320),490,1690
4,5,Roman,(1) Arizona,(1) Duke,260,220,(240),(240),(320),(320),480,1600
5,5,Penny,(1) Duke,(1) Arizona,240,240,(280),(240),(320),(320),480,1640
6,5,Arden,(1) Florida,(1) Michigan,240,240,(280),(240),(160),-,480,1160
7,5,Chad,(1) Michigan,(2) Houston,260,220,(240),(320),(320),(320),480,1680
8,9,Elaine,(1) Duke,(1) Arizona,250,220,(240),(320),(320),(320),470,1670
9,9,Sarah,(1) Florida,(1) Michigan,250,220,(240),(240),(160),-,470,1110


In [16]:
mwi.leaderboard['MWI Contestant'].tolist()

['Truett',
 'Eli',
 'Don',
 'Tim',
 'Roman',
 'Penny',
 'Arden',
 'Chad',
 'Elaine',
 'Sarah',
 'Maren',
 'Cynthia',
 'Emma',
 'August',
 'Henny',
 'Colombina',
 'Shawn',
 'Indie',
 'Mary',
 'Lucia',
 'Pete',
 'Sabine',
 'Winter',
 'Pia',
 'Chalamet']

### TODO: Make this a method in MWI

In [10]:
# (mwi.leaderboard.
#      style.hide_index()
#      .set_caption('2026 MWI Standings')
#      .set_table_styles([{
#             'selector': 'caption',
#             'props': [
#                 ('color', 'black'),
#                 ('font-size', '20px')
#             ]
#         }])
#     .export_png('2026_MWI_Standings.png', dpi=800)
# )

In [11]:
# df = mwi.leaderboard.copy()

# score_cols = [
#     "Place", "Round of 64", "Round of 32", "Sweet 16",
#     "Elite 8", "Final 4", "Championship",
#     "Total Points", "Total Potential Points"
# ]

# styled = (
#     df.style
#       .hide_index()
#       .format(na_rep="-")
#       .set_caption("2026 MWI Standings")
#       .set_table_styles([
#           {
#               "selector": "table",
#               "props": [
#                   ("border-collapse", "collapse"),
#                   ("font-family", "Arial, sans-serif"),
#                   ("font-size", "15px"),
#                   ("color", "#222"),
#               ],
#           },
#           {
#               "selector": "caption",
#               "props": [
#                   ("caption-side", "top"),
#                   ("font-size", "22px"),
#                   ("font-weight", "500"),
#                   ("color", "#222"),
#                   ("padding-bottom", "10px"),
#                   ("text-align", "center"),
#               ],
#           },
#           {
#               "selector": "thead th",
#               "props": [
#                   ("background-color", "#ffffff"),
#                   ("color", "#222"),
#                   ("font-weight", "600"),
#                   ("padding", "8px 12px"),
#                   ("border-bottom", "1px solid #d9d9d9"),
#                   ("text-align", "center"),
#                   ("white-space", "nowrap"),
#               ],
#           },
#           {
#               "selector": "tbody td",
#               "props": [
#                   ("padding", "10px 12px"),
#                   ("text-align", "center"),
#                   ("white-space", "nowrap"),
#                   ("color", "#222"),
#                   ("border", "none"),
#               ],
#           },
#           {
#               "selector": "tbody tr:nth-child(odd) td",
#               "props": [
#                   ("background-color", "#f3f3f3"),
#               ],
#           },
#           {
#               "selector": "tbody tr:nth-child(even) td",
#               "props": [
#                   ("background-color", "#e8e8e8"),
#               ],
#           },
#       ])
#       .set_properties(subset=["Place"], **{"min-width": "55px"})
#       .set_properties(subset=["MWI Contestant"], **{"min-width": "150px"})
#       .set_properties(subset=["Winner", "Runner-Up"], **{"min-width": "170px"})
#       .set_properties(
#           subset=["Round of 64", "Round of 32", "Sweet 16", "Elite 8", "Final 4"],
#           **{"min-width": "95px"}
#       )
#       .set_properties(subset=["Championship"], **{"min-width": "120px"})
#       .set_properties(
#           subset=["Total Points", "Total Potential Points"],
#           **{"min-width": "145px"}
#       )
# )

# dfi.export(
#     styled,
#     "2026_MWI_Standings.png",
#     max_rows=-1,
#     dpi=250,
#     table_conversion="playwright",
# )

In [12]:
# from html2image import Html2Image
# from PIL import Image, ImageChops

# df = mwi.leaderboard.copy().reset_index(drop=True)

# styled = (
#     df.style
#       .hide_index()   # or .hide_index() on older pandas
#       .format(na_rep="-")
#       .set_caption("2026 MWI Standings")
# )

# html = styled.to_html()

# css = """
# html, body {
#     margin: 0;
#     padding: 0;
#     background: #efefef;   /* make outer background uniform */
# }
# table {
#     border-collapse: collapse;
#     margin: 0;
#     font-family: Arial, sans-serif;
#     font-size: 15px;
#     background: white;
# }
# caption {
#     caption-side: top;
#     font-size: 22px;
#     padding: 8px 0 10px 0;
#     text-align: center;
# }
# thead th {
#     background: white;
#     color: black;
#     font-weight: 600;
#     padding: 8px 12px;
#     border-bottom: 1px solid #d9d9d9;
#     white-space: nowrap;
#     text-align: center;
# }
# tbody td {
#     padding: 10px 12px;
#     white-space: nowrap;
#     text-align: center;
#     color: black;
# }
# tbody tr:nth-child(odd) td  { background: #ffffff; }
# tbody tr:nth-child(even) td { background: #f3f3f3; }
# """

# hti = Html2Image(
#     size=(1800, 1200),  # closer to the table, so less to crop
#     custom_flags=["--default-background-color=efefef", "--hide-scrollbars"]
# )

# paths = hti.screenshot(
#     html_str=html,
#     css_str=css,
#     save_as="2026_MWI_Standings_raw.png",
# )

# # Auto-crop outer whitespace
# img = Image.open(paths[0]).convert("RGB")

# # Use the bottom-right pixel as the page background color
# bg_color = img.getpixel((img.width - 1, img.height - 1))
# bg = Image.new("RGB", img.size, bg_color)

# diff = ImageChops.difference(img, bg)
# bbox = diff.getbbox()

# if bbox:
#     pad = 8  # small border around the table
#     left, top, right, bottom = bbox
#     cropped = img.crop((
#         max(0, left - pad),
#         max(0, top - pad),
#         min(img.width, right + pad),
#         min(img.height, bottom + pad),
#     ))
#     cropped.save("2026_MWI_Standings.png")
# else:
#     img.save("2026_MWI_Standings.png")

In [17]:
from html2image import Html2Image
from PIL import Image, ImageChops

# -----------------------
# Tunables
# -----------------------
render_scale = 4          # 2 = crisper, 3 = even crisper but larger/slower
base_viewport = (1800, 1200)
page_bg = "#e6e6e6"
raw_file = "2026_MWI_Standings_raw.png"
final_file = "2026_MWI_Standings.png"

# -----------------------
# Data
# -----------------------
df = mwi.leaderboard.copy().reset_index(drop=True)

styler = df.style
if hasattr(styler, "hide"):
    styler = styler.hide_index()
else:
    styler = styler.hide_index()

styler = (
    styler
    .format(na_rep="-")
    .set_caption("2026 MWI Standings")
)

table_html = styler.to_html()

# -----------------------
# HTML wrapper
# -----------------------
html = f"""
<!doctype html>
<html>
<head>
    <meta charset="utf-8">
</head>
<body>
    <div id="capture">
        {table_html}
    </div>
</body>
</html>
"""

# -----------------------
# CSS
# -----------------------
css = f"""
html, body {{
    margin: 0;
    padding: 0;
    background: {page_bg};
}}

#capture {{
    display: inline-block;
    transform: scale({render_scale});
    transform-origin: top left;
}}

table {{
    border-collapse: collapse;
    margin: 0;
    background: #ffffff;
    font-family: Arial, Helvetica, sans-serif;
    font-size: 15px;
    color: #222222;
}}

caption {{
    caption-side: top;
    font-size: 22px;
    font-weight: 500;
    text-align: center;
    padding: 8px 0 10px 0;
    color: #222222;
}}

thead th {{
    background: #ffffff;
    color: #222222;
    font-weight: 600;
    padding: 8px 12px;
    border-bottom: 1px solid #d9d9d9;
    white-space: nowrap;
    text-align: center;
}}

tbody td {{
    padding: 10px 12px;
    white-space: nowrap;
    text-align: center;
    color: #222222;
}}

tbody tr:nth-child(odd) td {{
    background: #ffffff;
}}

tbody tr:nth-child(even) td {{
    background: #f3f3f3;
}}
"""

# -----------------------
# Render oversized screenshot
# -----------------------
hti = Html2Image(
    size=(base_viewport[0] * render_scale, base_viewport[1] * render_scale),
    custom_flags=[
        "--default-background-color=e6e6e6",
        "--hide-scrollbars",
    ],
)

paths = hti.screenshot(
    html_str=html,
    css_str=css,
    save_as=raw_file,
)

# -----------------------
# Crop excess whitespace
# -----------------------
img = Image.open(paths[0]).convert("RGB")

bg_color = img.getpixel((img.width - 1, img.height - 1))
bg = Image.new("RGB", img.size, bg_color)
diff = ImageChops.difference(img, bg)
bbox = diff.getbbox()

if bbox:
    pad = 8 * render_scale
    left, top, right, bottom = bbox
    img = img.crop((
        max(0, left - pad),
        max(0, top - pad),
        min(img.width, right + pad),
        min(img.height, bottom + pad),
    ))

# -----------------------
# Downsample for sharper final output
# -----------------------
if hasattr(Image, "Resampling"):
    resample_method = Image.Resampling.LANCZOS
else:
    resample_method = Image.LANCZOS

final_img = img.resize(
    (img.width // render_scale, img.height // render_scale),
    resample_method
)

final_img.save(final_file)
print(f"Saved {final_file}")

[44124:7860240:0323/085701.490068:ERROR:ui/display/mac/cv_display_link_mac.mm:168] CVDisplayLinkCreateWithCGDisplay failed. CVReturn: -6670
[44124:7860320:0323/085701.490765:ERROR:ui/display/mac/cv_display_link_mac.mm:168] CVDisplayLinkCreateWithCGDisplay failed. CVReturn: -6670
[44124:7860240:0323/085701.491021:ERROR:ui/display/mac/cv_display_link_mac.mm:168] CVDisplayLinkCreateWithCGDisplay failed. CVReturn: -6670
[44124:7860240:0323/085701.491561:ERROR:ui/display/mac/cv_display_link_mac.mm:168] CVDisplayLinkCreateWithCGDisplay failed. CVReturn: -6670
[44124:7860320:0323/085701.491657:ERROR:ui/display/mac/cv_display_link_mac.mm:168] CVDisplayLinkCreateWithCGDisplay failed. CVReturn: -6670
[44124:7860240:0323/085701.491755:ERROR:ui/display/mac/cv_display_link_mac.mm:168] CVDisplayLinkCreateWithCGDisplay failed. CVReturn: -6670
[44124:7860240:0323/085701.492227:ERROR:ui/display/mac/cv_display_link_mac.mm:168] CVDisplayLinkCreateWithCGDisplay failed. CVReturn: -6670
[44124:7860320:0323/

Saved 2026_MWI_Standings.png
